# Music Annotation Project — Notebook 37
## V8.2h Downward PRINT-Threshold Safe-Coverage Frontier — Catastrophic Audit Amendment

**Runtime:** CPU is sufficient. Run all cells in order.

This is a **development-only, post-outcome threshold-sensitivity experiment**. It does not retrain, refit or recalibrate V8.2. It reuses the exact saved source-group OOF probabilities from one completed V8.2 run and asks:

> As the coordinated PRINT probability/margin threshold is lowered, how much additional recoverable print is preserved, where does false annotation enter, and at which threshold bands does the practical whole-page result change?

### Design

- Registered comparator: effective PRINT threshold `0.995`.
- Coordinated downward sweep: threshold `t`, margin `2t - 1`; for the binary V8.2 model this makes the effective decision boundary exactly `p_print >= t`.
- Page-global connected-run filter remains on with minimum run size 2.
- Every candidate page starts from frozen V7.8 and restores the exact observed byte only at final PRINT coordinates.
- The `0.995` reconstruction must replay the original V8.2 page exactly.
- Metrics are reported cumulatively and for each **newly added threshold band**.
- Uncertainty uses 10,000 page-cluster bootstrap resamples plus Wilson upper bounds.
- Review artefacts include complete pages and both local and wide-context crops, specifically to avoid judging a lower staff/system from an over-tight crop.
- Every truth-defined catastrophic component receives a first-appearance transition audit, cumulative and incremental truth overlays, a whole-page locator, contact sheets and a durable visual-review queue.

### Interpretation boundary

This notebook can identify promising development thresholds and the first threshold band where safety deteriorates. It cannot freeze or promote a new policy. Notebook 36 motivates the question but is not used to fit or choose the answer. A selected policy would require a separately declared challenger and fresh validation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT = ROOT / 'observed_print_classifier_v8_2'
V82G_ROOT = ROOT / 'observed_print_classifier_v8_2g_blinded_group_review'
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v8_2h_threshold_frontier'

# Optional exact source run. None selects the latest complete development-only V8.2 run.
SOURCE_V82_RUN_OVERRIDE = None

# Coordinated effective thresholds. Keep descending and include the registered 0.995 comparator.
SWEEP_THRESHOLDS = [
    0.995, 0.990, 0.985, 0.980, 0.975,
    0.950, 0.925, 0.900, 0.875, 0.850,
    0.825, 0.800, 0.750, 0.700, 0.600, 0.500,
]
ANNOTATION_THRESHOLD = 0.995
ENABLE_RUN_FILTER = True
MIN_PRINT_RUN = 2
BOOTSTRAP_RESAMPLES = 10000
BOOTSTRAP_SEED = 37001

# Whole-page outputs are generated only for these thresholds.
RENDER_THRESHOLDS = [0.995, 0.980, 0.950, 0.900, 0.850, 0.800]
OVERLAY_RADIUS = 1
OVERLAY_ALPHA = 0.62
LOCAL_CONTEXT_PADDING = 80
WIDE_CONTEXT_PADDING = 260
MAX_REVIEW_COMPONENTS_PER_INCREMENT = 60
CATASTROPHE_LOCAL_PADDING = 80
CATASTROPHE_WIDE_PADDING = 260
CATASTROPHE_PAGE_MAX_WIDTH = 1200
CATASTROPHE_CONTACT_SHEET_ROWS = 6

if SWEEP_THRESHOLDS != sorted(set(SWEEP_THRESHOLDS), reverse=True):
    raise ValueError('SWEEP_THRESHOLDS must be unique and strictly descending.')
if SWEEP_THRESHOLDS[0] != 0.995:
    raise ValueError('The first threshold must be the registered 0.995 comparator.')
if not set(RENDER_THRESHOLDS).issubset(SWEEP_THRESHOLDS):
    raise ValueError('Every RENDER_THRESHOLDS value must appear in SWEEP_THRESHOLDS.')
if MIN_PRINT_RUN < 1:
    raise ValueError('MIN_PRINT_RUN must be at least 1.')

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_2h_threshold_frontier'
OUT = OUTPUT_ROOT / RUN
TABLES = OUT / 'tables'
WHOLE = OUT / 'whole_pages'
VISUAL = OUT / 'visual_audit'
AUDIT = OUT / 'audit'
REVIEW = OUT / 'review'
for p in [OUT, TABLES, WHOLE, VISUAL, AUDIT, REVIEW]:
    p.mkdir(parents=True, exist_ok=False)
print('Output:', OUT)

## 2. Resolve and verify the exact V8.2 development source

In [ ]:
REQUIRED_V82 = [
    'experiment_identity.json',
    'tables/all_eligible_oof_predictions.parquet',
    'whole_pages',
]

def complete_runs(root, required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):
                hits.append(p)
    return sorted(hits, key=lambda p:p.stat().st_mtime)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''):
            h.update(block)
    return h.hexdigest()

if SOURCE_V82_RUN_OVERRIDE is not None:
    SOURCE = Path(SOURCE_V82_RUN_OVERRIDE)
    missing=[r for r in REQUIRED_V82 if not (SOURCE/r).exists()]
    if missing:
        raise RuntimeError(f'Incomplete V8.2 source override: {missing}')
else:
    hits=complete_runs(V82_ROOT, REQUIRED_V82)
    if not hits:
        raise RuntimeError(f'No complete V8.2 development run below {V82_ROOT}')
    SOURCE=hits[-1]

source_identity=json.loads((SOURCE/'experiment_identity.json').read_text())
if source_identity.get('version')!='8.2-development-whole-page':
    raise RuntimeError(f"Unexpected V8.2 source version: {source_identity.get('version')}")
if source_identity.get('independent_data_used') is not False:
    raise RuntimeError('Notebook 37 may use only the development-only V8.2 OOF run.')

PRED_PATH=SOURCE/'tables/all_eligible_oof_predictions.parquet'
full=pd.read_parquet(PRED_PATH).reset_index(drop=True)
required={'page','source_group','y','x','truth','fold','raw_print','p_print'}
missing=required-set(full.columns)
if missing:
    raise RuntimeError(f'V8.2 predictions missing columns: {sorted(missing)}')
full['page']=full.page.astype(str)
full['source_group']=full.source_group.astype(str)
for c in ['y','x','truth','fold']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
for c in ['raw_print','p_print']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(np.float64)
if not np.isfinite(full[['raw_print','p_print']].to_numpy()).all():
    raise RuntimeError('Non-finite V8.2 probability values.')
if full.duplicated(['page','y','x']).any():
    raise RuntimeError('Duplicate page/y/x prediction rows.')
if not full.truth.isin([0,1,2]).all():
    raise RuntimeError('Unexpected truth labels.')

source_record={
    'source_v82_run':str(SOURCE),
    'source_identity_sha256':sha256_file(SOURCE/'experiment_identity.json'),
    'source_predictions_sha256':sha256_file(PRED_PATH),
    'eligible_rows':int(len(full)),
    'pages':int(full.page.nunique()),
    'source_groups':int(full.source_group.nunique()),
}
(OUT/'source_v82_record.json').write_text(json.dumps(source_record,indent=2))
print(json.dumps(source_record,indent=2))

## 3. Load the authoritative mask/component context

The component identity comes from the exact V7.5/V7.8 reference named by the V8.2 source. It is used only for connected-component safety accounting and integrity checks.

In [ ]:
v75=Path(source_identity['v7_5_reference'])
status=pd.read_csv(v75/'tables/cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
page_context={}
for page in sorted(full.page.unique()):
    if page not in cache:
        raise RuntimeError(f'No V7.5 cache for {page}')
    with np.load(cache[page]) as z:
        if 'mask' not in z.files:
            raise RuntimeError(f'{page}: cache lacks mask')
        mask=z['mask']>0
    n,labels,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    page_context[page]={
        'mask':mask,
        'component_id':labels.astype(np.int32),
        'component_stats':stats,
    }
print('Loaded authoritative context for',len(page_context),'pages')

## 4. Coordinated policy and page-global connected-run helpers

In [ ]:
def policy_id(threshold):
    return f'p{threshold:.3f}_m{max(0.0,2*threshold-1):.3f}_run{MIN_PRINT_RUN if ENABLE_RUN_FILTER else 1}'

def provisional_decision(prob, threshold):
    p=np.asarray(prob,dtype=np.float64)
    q=1.0-p
    margin=max(0.0,2.0*float(threshold)-1.0)
    out=np.full(len(p),'UNCERTAIN',object)
    out[q>=ANNOTATION_THRESHOLD]='ANNOTATION'
    out[(p>=threshold)&((p-q)>=margin)]='PRINT'
    return out

def page_global_run_filter(frame, provisional):
    final=np.asarray(provisional,dtype=object).copy()
    run_size=np.zeros(len(frame),np.int32)
    run_id=np.zeros(len(frame),np.int32)
    demoted=0
    if not ENABLE_RUN_FILTER or MIN_PRINT_RUN<=1:
        return final,run_size,run_id,demoted
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy()
        ctx=page_context[page]
        pm=np.zeros(ctx['mask'].shape,np.uint8)
        candidate=provisional[idx]=='PRINT'
        ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
        pm[ys[candidate],xs[candidate]]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
        labels=lab[ys,xs]
        sizes=np.zeros(len(g),np.int32)
        valid=labels>0
        sizes[valid]=stats[labels[valid],cv2.CC_STAT_AREA]
        run_size[idx]=sizes
        run_id[idx]=labels
        drop=candidate & (sizes<MIN_PRINT_RUN)
        final[idx[drop]]='UNCERTAIN'
        demoted+=int(drop.sum())
    return final,run_size,run_id,demoted

def wilson_upper(errors,n,z=1.96):
    if n==0:
        return np.nan
    p=errors/n; d=1+z*z/n
    centre=p+z*z/(2*n)
    radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return (centre+radius)/d

print('Policy helpers ready.')

## 5. Evaluate each cumulative threshold policy

In [ ]:
def evaluate_threshold(frame, threshold, decisions, run_size, run_id, demoted):
    pid=policy_id(threshold)
    work=frame[['page','source_group','y','x','truth','p_print']].copy()
    work['decision']=decisions
    work['run_size']=run_size
    work['run_id']=run_id
    selected=work.decision.eq('PRINT')

    comp_rows=[]
    for page,g in work[selected].groupby('page',sort=False):
        ctx=page_context[page]
        cids=ctx['component_id'][g.y.to_numpy(int),g.x.to_numpy(int)]
        g=g.copy(); g['mask_component_id']=cids
        for cid,q in g[g.mask_component_id>0].groupby('mask_component_id'):
            tp=int((q.truth==1).sum()); fp=int((q.truth==0).sum()); tu=int((q.truth==2).sum())
            catastrophic=bool(fp>=50 or (fp/max(tp,1)>1.0 and fp>=20))
            ys=q.y.to_numpy(int); xs=q.x.to_numpy(int)
            comp_rows.append({
                'policy_id':pid,'threshold':threshold,'page':page,
                'source_group':q.source_group.iloc[0],'mask_component_id':int(cid),
                'selected':int(len(q)),'true_print':tp,'false_annotation':fp,
                'truth_uncertain':tu,'utility_without_catastrophe':tp-5*fp,
                'catastrophic':catastrophic,
                'xmin':int(xs.min()),'xmax':int(xs.max()),
                'ymin':int(ys.min()),'ymax':int(ys.max()),
                'mean_p_print':float(q.p_print.mean()),'min_p_print':float(q.p_print.min()),
            })
    comp_cols=['policy_id','threshold','page','source_group','mask_component_id','selected','true_print','false_annotation','truth_uncertain','utility_without_catastrophe','catastrophic','xmin','xmax','ymin','ymax','mean_p_print','min_p_print']
    comps=pd.DataFrame(comp_rows,columns=comp_cols)

    page_rows=[]
    for page,g in work.groupby('page',sort=False):
        s=g.decision.eq('PRINT')
        tp=int((s&(g.truth==1)).sum()); fp=int((s&(g.truth==0)).sum()); tu=int((s&(g.truth==2)).sum())
        cats=int(((comps.page==page)&comps.catastrophic).sum()) if len(comps) else 0
        page_rows.append({
            'policy_id':pid,'threshold':threshold,'page':page,
            'source_group':g.source_group.iloc[0],'eligible':int(len(g)),
            'selected':int(s.sum()),'true_print':tp,'false_annotation':fp,
            'truth_uncertain_print':tu,'catastrophic_components':cats,
            'utility':tp-5*fp-500*cats,
            'print_coverage':float(s.mean()),
            'uncertain_rate':float(g.decision.eq('UNCERTAIN').mean()),
        })
    pages=pd.DataFrame(page_rows)
    source=(pages.groupby('source_group',as_index=False).agg(
        pages=('page','nunique'),selected=('selected','sum'),true_print=('true_print','sum'),
        false_annotation=('false_annotation','sum'),truth_uncertain_print=('truth_uncertain_print','sum'),
        catastrophic_components=('catastrophic_components','sum'),utility=('utility','sum'),
    ))
    source['false_true_ratio']=source.false_annotation/np.maximum(source.true_print,1)

    tp=int(pages.true_print.sum()); fp=int(pages.false_annotation.sum())
    tu=int(pages.truth_uncertain_print.sum()); cats=int(pages.catastrophic_components.sum())
    selected_n=int(pages.selected.sum()); selected_scored=tp+fp
    benefit_pages=int((pages.true_print>=10).sum())
    benefit_groups=int(pages.loc[pages.true_print>=10,'source_group'].nunique())

    # Vectorised page-cluster bootstrap.
    vals=pages[['true_print','false_annotation','truth_uncertain_print','catastrophic_components']].to_numpy(np.float64)
    rng=np.random.default_rng(BOOTSTRAP_SEED)
    samples=[]
    remaining=BOOTSTRAP_RESAMPLES
    while remaining:
        batch=min(1000,remaining)
        idx=rng.integers(0,len(vals),size=(batch,len(vals)))
        totals=vals[idx].sum(axis=1)
        bt,bf,bu,bc=totals.T
        utility=bt-5*bf-500*bc
        false_true=bf/np.maximum(bt,1)
        false_pres=bf/np.maximum(bt+bf+bu,1)
        samples.append(np.column_stack([utility,false_true,false_pres]))
        remaining-=batch
    boot=np.vstack(samples)

    stable_source=source[source.pages>=3]
    source_stability=bool(((stable_source.false_true_ratio<=0.25)&(stable_source.catastrophic_components==0)).all())
    summary={
        'policy_id':pid,'threshold':float(threshold),'print_margin':float(max(0,2*threshold-1)),
        'annotation_threshold':ANNOTATION_THRESHOLD,'run_filter':ENABLE_RUN_FILTER,
        'minimum_print_run':MIN_PRINT_RUN,'run_filter_demotions':int(demoted),
        'eligible_pixels':int(len(work)),'selected_print_pixels':selected_n,
        'true_print':tp,'false_annotation':fp,'truth_uncertain_print':tu,
        'false_true_ratio':fp/max(tp,1),
        'false_preservation_rate':fp/max(selected_n,1),
        'wilson95_upper_false_preservation':wilson_upper(fp,selected_n),
        'utility':tp-5*fp-500*cats,'catastrophic_components':cats,
        'benefiting_pages':benefit_pages,'benefiting_source_groups':benefit_groups,
        'print_coverage':selected_n/max(len(work),1),
        'uncertain_rate':float(work.decision.eq('UNCERTAIN').mean()),
        'bootstrap_utility_95_low':float(np.quantile(boot[:,0],.025)),
        'bootstrap_utility_95_high':float(np.quantile(boot[:,0],.975)),
        'bootstrap_false_true_95_high':float(np.quantile(boot[:,1],.975)),
        'bootstrap_false_preservation_95_high':float(np.quantile(boot[:,2],.975)),
        'source_stability_for_groups_with_3_pages':source_stability,
    }
    summary.update({
        'D3_catastrophic_safety':cats==0,
        'D4_false_preservation':(summary['false_true_ratio']<=0.10 and summary['bootstrap_false_true_95_high']<=0.20),
        'D5_pixel_safety':(summary['false_preservation_rate']<=0.005 and summary['wilson95_upper_false_preservation']<=0.01),
        'D6_positive_utility':(summary['utility']>0 and summary['bootstrap_utility_95_low']>0),
        'D7_nontrivial_benefit':(tp>=500 and benefit_pages>=10 and benefit_groups>=3),
        'D8_source_stability':source_stability,
    })
    summary['core_development_envelope_D3_to_D8']=all(summary[k] for k in [
        'D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety',
        'D6_positive_utility','D7_nontrivial_benefit','D8_source_stability'])
    return summary,pages,source,comps,work,boot

results={}; summary_rows=[]; page_rows=[]; source_rows=[]; component_rows=[]
for t in SWEEP_THRESHOLDS:
    provisional=provisional_decision(full.p_print.to_numpy(),t)
    decisions,run_size,run_id,demoted=page_global_run_filter(full,provisional)
    summary,pages,source,comps,work,boot=evaluate_threshold(full,t,decisions,run_size,run_id,demoted)
    results[t]={'summary':summary,'pages':pages,'source':source,'components':comps,'work':work,'boot':boot}
    summary_rows.append(summary); page_rows.append(pages); source_rows.append(source.assign(policy_id=policy_id(t),threshold=t))
    if len(comps): component_rows.append(comps)

sweep=pd.DataFrame(summary_rows).sort_values('threshold',ascending=False).reset_index(drop=True)
sweep.to_csv(TABLES/'threshold_frontier_summary.csv',index=False)
pd.concat(page_rows,ignore_index=True).to_csv(TABLES/'threshold_frontier_by_page.csv',index=False)
pd.concat(source_rows,ignore_index=True).to_csv(TABLES/'threshold_frontier_by_source_group.csv',index=False)
pd.concat(component_rows,ignore_index=True).to_csv(TABLES/'threshold_frontier_components.csv',index=False)
print(sweep[['threshold','selected_print_pixels','true_print','false_annotation','truth_uncertain_print','utility','catastrophic_components','core_development_envelope_D3_to_D8']].to_string(index=False))

## 6. Incremental threshold-band accounting

Each row below isolates only pixels newly selected when moving from the previous higher threshold to the next lower threshold. Because the connected-run rule can cause a newly connected run to enter together, `p_print` values can extend above the nominal lower band; the exact higher/lower policy membership is authoritative.

In [ ]:
increment_rows=[]; increment_pixel_rows=[]; review_component_rows=[]
for hi,lo in zip(SWEEP_THRESHOLDS[:-1],SWEEP_THRESHOLDS[1:]):
    high=results[hi]['work']; low=results[lo]['work']
    hs=high.decision.eq('PRINT').to_numpy(); ls=low.decision.eq('PRINT').to_numpy()
    if np.any(hs & ~ls):
        raise RuntimeError(f'Non-monotone final PRINT set between {hi} and {lo}')
    new=ls & ~hs
    q=low.loc[new].copy()
    tp=int((q.truth==1).sum()); fp=int((q.truth==0).sum()); tu=int((q.truth==2).sum())
    increment_id=f'new_{lo:.3f}_vs_{hi:.3f}'
    increment_rows.append({
        'increment_id':increment_id,'higher_threshold':hi,'lower_threshold':lo,
        'new_selected_pixels':int(len(q)),'new_true_print':tp,'new_false_annotation':fp,
        'new_truth_uncertain':tu,'incremental_utility_without_catastrophe':tp-5*fp,
        'pages':int(q.page.nunique()),'source_groups':int(q.source_group.nunique()),
        'mean_p_print':float(q.p_print.mean()) if len(q) else np.nan,
        'median_p_print':float(q.p_print.median()) if len(q) else np.nan,
    })
    if len(q):
        q['increment_id']=increment_id; q['higher_threshold']=hi; q['lower_threshold']=lo
        increment_pixel_rows.append(q)
        # Group new pixels by authoritative mask component for contextual review.
        for page,g in q.groupby('page',sort=False):
            cids=page_context[page]['component_id'][g.y.to_numpy(int),g.x.to_numpy(int)]
            g=g.copy(); g['mask_component_id']=cids
            for cid,z in g[g.mask_component_id>0].groupby('mask_component_id'):
                ys=z.y.to_numpy(int); xs=z.x.to_numpy(int)
                ctp=int((z.truth==1).sum()); cfp=int((z.truth==0).sum()); ctu=int((z.truth==2).sum())
                review_component_rows.append({
                    'increment_id':increment_id,'higher_threshold':hi,'lower_threshold':lo,
                    'page':page,'source_group':z.source_group.iloc[0],
                    'mask_component_id':int(cid),'new_pixels':int(len(z)),
                    'new_true_print':ctp,'new_false_annotation':cfp,'new_truth_uncertain':ctu,
                    'incremental_utility':ctp-5*cfp,
                    'xmin':int(xs.min()),'xmax':int(xs.max()),'ymin':int(ys.min()),'ymax':int(ys.max()),
                    'mean_p_print':float(z.p_print.mean()),'min_p_print':float(z.p_print.min()),
                })

increments=pd.DataFrame(increment_rows)
increments.to_csv(TABLES/'incremental_threshold_band_summary.csv',index=False)
increment_pixels=pd.concat(increment_pixel_rows,ignore_index=True) if increment_pixel_rows else pd.DataFrame()
increment_pixels.to_parquet(TABLES/'incremental_threshold_pixels.parquet',index=False)
review_components=pd.DataFrame(review_component_rows)
review_components.to_csv(TABLES/'incremental_component_review_queue_all.csv',index=False)
print(increments.to_string(index=False))

## 7. Diagnostic frontier plots and candidate threshold report

In [ ]:
q=sweep.sort_values('threshold')
plt.figure(figsize=(10,6))
plt.plot(q.threshold,q.true_print,marker='o',label='True PRINT')
plt.plot(q.threshold,q.false_annotation,marker='x',label='False annotation')
plt.xlabel('Effective PRINT threshold'); plt.ylabel('Selected pixels')
plt.title('Cumulative threshold frontier'); plt.legend(); plt.tight_layout()
plt.savefig(VISUAL/'threshold_frontier_true_false.png',dpi=180); plt.close()

plt.figure(figsize=(10,6))
plt.plot(q.threshold,q.utility,marker='o',label='Net safety utility')
plt.fill_between(q.threshold,q.bootstrap_utility_95_low,q.bootstrap_utility_95_high,alpha=.2,label='Page-bootstrap 95% interval')
plt.axhline(0,color='black',linewidth=1)
plt.xlabel('Effective PRINT threshold'); plt.ylabel('Utility')
plt.title('Utility across downward PRINT threshold sweep'); plt.legend(); plt.tight_layout()
plt.savefig(VISUAL/'threshold_frontier_utility.png',dpi=180); plt.close()

plt.figure(figsize=(10,6))
plt.plot(q.threshold,q.print_coverage,marker='o',label='PRINT coverage')
plt.plot(q.threshold,q.false_preservation_rate,marker='x',label='False-preservation rate')
plt.axhline(.005,color='red',linestyle='--',linewidth=1,label='D5 observed ceiling 0.5%')
plt.xlabel('Effective PRINT threshold'); plt.ylabel('Rate')
plt.title('Coverage and false-preservation rate'); plt.legend(); plt.tight_layout()
plt.savefig(VISUAL/'threshold_frontier_coverage_safety.png',dpi=180); plt.close()

passing=sweep[sweep.core_development_envelope_D3_to_D8]
lowest_core=float(passing.threshold.min()) if len(passing) else None
best_utility=sweep.sort_values(['utility','true_print'],ascending=False).iloc[0].to_dict()
report={
    'lowest_threshold_meeting_computable_core_D3_to_D8':lowest_core,
    'highest_observed_utility_policy':best_utility,
    'important_limit':'D1, D2, D9, D10, D11 and D12 are not fully adjudicated by this shortcut; no row is promotion-authorised.',
}
(OUT/'diagnostic_frontier_report.json').write_text(json.dumps(report,indent=2,default=str))
print(json.dumps(report,indent=2,default=str))

## 8. Render whole pages at selected thresholds and prove strict replay

For each rendered threshold, the candidate page starts from V7.8 and copies exact observed bytes only at that threshold's final PRINT coordinates. The registered `0.995` candidate must be byte-identical to the saved V8.2 source page.

In [ ]:
MAGENTA=(255,0,255)
def overlay_points(gray,mask,colour=MAGENTA,radius=OVERLAY_RADIUS,alpha=OVERLAY_ALPHA):
    active=mask.copy()
    if radius>0:
        active=cv2.dilate(active.astype(np.uint8),np.ones((2*radius+1,2*radius+1),np.uint8))>0
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); layer=base.copy(); layer[active]=colour
    out=base.copy(); out[active]=cv2.addWeighted(base,1-alpha,layer,alpha,0)[active]
    return out

def threshold_slug(t): return f'p{int(round(t*1000)):04d}'

integrity=[]
for t in RENDER_THRESHOLDS:
    work=results[t]['work']
    strict_work=results[0.995]['work']
    for page,g in work.groupby('page',sort=False):
        src=SOURCE/'whole_pages'/page
        observed=cv2.imread(str(src/'observed.png'),0)
        v78=cv2.imread(str(src/'v7_8.png'),0)
        v82=cv2.imread(str(src/'v8_2.png'),0)
        if observed is None or v78 is None or v82 is None or not (observed.shape==v78.shape==v82.shape):
            raise RuntimeError(f'{page}: missing or mismatched V8.2 whole-page inputs')
        ys=g.y.to_numpy(int); xs=g.x.to_numpy(int); selected=g.decision.eq('PRINT').to_numpy()
        pm=np.zeros(observed.shape,bool); pm[ys[selected],xs[selected]]=True
        out=v78.copy(); out[pm]=observed[pm]
        auth=page_context[page]['mask']
        outside=int(np.sum(out[~auth]!=observed[~auth]))
        nonprint=int(np.sum(out[~pm]!=v78[~pm]))
        wrong=int(np.sum((out!=v78)&(out!=observed)))
        if outside or nonprint or wrong:
            raise RuntimeError(f'{t}/{page}: page integrity failure')
        if t==0.995 and not np.array_equal(out,v82):
            raise RuntimeError(f'{page}: registered 0.995 policy does not replay source V8.2 exactly')
        sg=strict_work[strict_work.page.eq(page)]
        sys=sg.y.to_numpy(int); sxs=sg.x.to_numpy(int); ssel=sg.decision.eq('PRINT').to_numpy()
        strict_pm=np.zeros(observed.shape,bool); strict_pm[sys[ssel],sxs[ssel]]=True
        newly=pm&~strict_pm
        d=WHOLE/threshold_slug(t)/page; d.mkdir(parents=True,exist_ok=False)
        diff=np.zeros_like(observed); diff[out!=v82]=255
        cv2.imwrite(str(d/'observed.png'),observed)
        cv2.imwrite(str(d/'v7_8.png'),v78)
        cv2.imwrite(str(d/'v8_2_registered.png'),v82)
        cv2.imwrite(str(d/'threshold_candidate.png'),out)
        cv2.imwrite(str(d/'difference_from_registered_v8_2.png'),diff)
        cv2.imwrite(str(d/'newly_preserved_vs_0995_overlay.png'),overlay_points(observed,newly))
        integrity.append({
            'threshold':t,'page':page,'selected_pixels':int(pm.sum()),
            'new_vs_0995':int(newly.sum()),'outside_mask_drift':outside,
            'non_print_v78_regression':nonprint,'changed_not_equal_observed':wrong,
            'strict_replay_equal':bool(np.array_equal(out,v82)) if t==0.995 else None,
            'candidate_sha256':sha256_file(d/'threshold_candidate.png'),
        })
pd.DataFrame(integrity).to_csv(AUDIT/'whole_page_integrity_and_hashes.csv',index=False)
print('Rendered threshold/page combinations:',len(integrity))

## 9. Wide-context review queue and crop artefacts

The queue is balanced across incremental bands and prioritises false-heavy, uncertain-heavy and high-utility components. Every selected component gets:

- a local comparison;
- a wide-context comparison using `WIDE_CONTEXT_PADDING`;
- the complete higher-threshold and lower-threshold pages in the threshold folders.

This prevents a judgement from depending solely on a crop that excludes the related staff/system.

In [ ]:
def crop_box(image,x0,y0,x1,y1,pad):
    h,w=image.shape[:2]
    xa=max(0,int(x0)-pad); xb=min(w,int(x1)+pad+1)
    ya=max(0,int(y0)-pad); yb=min(h,int(y1)+pad+1)
    return image[ya:yb,xa:xb],(xa,ya,xb,yb)

def panel(images,titles,header=70):
    heights=[im.shape[0] for im in images]; widths=[im.shape[1] for im in images]
    H=max(heights); W=max(widths)
    canvas=np.full((H+header,W*len(images),3),255,np.uint8)
    for i,(im,title) in enumerate(zip(images,titles)):
        z=cv2.cvtColor(im,cv2.COLOR_GRAY2BGR) if im.ndim==2 else im
        canvas[header:header+z.shape[0],i*W:i*W+z.shape[1]]=z
        cv2.putText(canvas,title,(i*W+8,28),cv2.FONT_HERSHEY_SIMPLEX,.58,(25,25,25),1,cv2.LINE_AA)
    return canvas

queue_parts=[]
if len(review_components):
    for increment_id,g in review_components.groupby('increment_id',sort=False):
        ranked=pd.concat([
            g.sort_values(['new_false_annotation','new_pixels'],ascending=False).head(MAX_REVIEW_COMPONENTS_PER_INCREMENT//3),
            g.sort_values(['new_truth_uncertain','new_pixels'],ascending=False).head(MAX_REVIEW_COMPONENTS_PER_INCREMENT//3),
            g.sort_values(['incremental_utility','new_true_print'],ascending=False).head(MAX_REVIEW_COMPONENTS_PER_INCREMENT//3),
        ]).drop_duplicates(['page','mask_component_id'])
        queue_parts.append(ranked)
review_queue=pd.concat(queue_parts,ignore_index=True) if queue_parts else pd.DataFrame()
if len(review_queue):
    review_queue['review_order']=np.arange(1,len(review_queue)+1)
    review_queue['review_status']='unreviewed'
    review_queue['review_decision']=''
    review_queue['review_note']=''
    review_queue.to_csv(REVIEW/'threshold_increment_review_queue.csv',index=False)

    crop_inventory=[]
    for _,r in review_queue.iterrows():
        page=str(r.page); hi=float(r.higher_threshold); lo=float(r.lower_threshold)
        hi_path=WHOLE/threshold_slug(hi)/page/'threshold_candidate.png'
        lo_path=WHOLE/threshold_slug(lo)/page/'threshold_candidate.png'
        # If an adjacent threshold was not selected for full rendering, reconstruct in memory.
        def candidate_for(t):
            src=SOURCE/'whole_pages'/page
            observed=cv2.imread(str(src/'observed.png'),0); v78=cv2.imread(str(src/'v7_8.png'),0)
            g=results[t]['work']; g=g[g.page.eq(page)]
            ys=g.y.to_numpy(int); xs=g.x.to_numpy(int); sel=g.decision.eq('PRINT').to_numpy()
            out=v78.copy(); out[ys[sel],xs[sel]]=observed[ys[sel],xs[sel]]
            return observed,out
        observed,hi_img=candidate_for(hi); _,lo_img=candidate_for(lo)
        delta=(lo_img!=hi_img).astype(np.uint8)*255
        for label,pad in [('local',LOCAL_CONTEXT_PADDING),('wide',WIDE_CONTEXT_PADDING)]:
            crops=[]; box=None
            for im in [observed,hi_img,lo_img,delta]:
                c,box=crop_box(im,r.xmin,r.ymin,r.xmax,r.ymax,pad); crops.append(c)
            image=panel(crops,['Observed',f'Higher {hi:.3f}',f'Lower {lo:.3f}','New difference'])
            dest=REVIEW/'crops'/r.increment_id/label
            dest.mkdir(parents=True,exist_ok=True)
            path=dest/f'{page}_component_{int(r.mask_component_id)}.png'
            cv2.imwrite(str(path),image)
            crop_inventory.append({'review_order':int(r.review_order),'increment_id':r.increment_id,'page':page,'mask_component_id':int(r.mask_component_id),'context':label,'padding':pad,'crop_box':'|'.join(map(str,box)),'file':str(path.relative_to(OUT))})
    pd.DataFrame(crop_inventory).to_csv(REVIEW/'threshold_increment_crop_inventory.csv',index=False)
print('Review components:',len(review_queue))

## 10. Catastrophic-component first-appearance audit

A catastrophic component is defined by the unchanged registered rule already used above:

- at least 50 cumulative false-annotation PRINT pixels; or
- false/true ratio greater than 1.0 with at least 20 cumulative false-annotation PRINT pixels.

This section does not alter that definition. For every page/component pair that becomes catastrophic anywhere in the sweep, it records the **first lower threshold at which the rule becomes true** and the immediately preceding higher threshold. It then creates:

- local and wide transition panels;
- a whole-page locator;
- cumulative and newly-added truth overlays;
- a contact sheet ordered by first appearance and false-annotation count;
- a durable visual-review CSV with `Visually harmful`, `Visually neutral`, `Visually beneficial` and `Unclear` as permitted judgements.

The initial quality judgement should compare the higher- and lower-threshold corrected outputs. Clean truth and colour overlays are diagnostic evidence, not inference inputs or automatic policy overrides.

In [ ]:
# Stable truth colours in OpenCV BGR.
TRUTH_BGR = {1:(0,200,0), 0:(0,0,230), 2:(0,180,255)}  # PRINT green, ANNOTATION red, UNCERTAIN amber
PURPLE = (255,0,255)
CATASTROPHE_REVIEW_CHOICES = ['Visually harmful','Visually neutral','Visually beneficial','Unclear']


def image_map(folder):
    supported={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in supported} if folder.exists() else {}


CLEAN_MAP=image_map(ROOT/'source'/'clean')
missing_clean=sorted(set(full.page.astype(str))-set(CLEAN_MAP))
if missing_clean:
    raise RuntimeError(f'Catastrophic audit lacks clean diagnostics for pages: {missing_clean[:10]}')


def candidate_for_threshold(page,t):
    src=SOURCE/'whole_pages'/str(page)
    observed=cv2.imread(str(src/'observed.png'),cv2.IMREAD_GRAYSCALE)
    v78=cv2.imread(str(src/'v7_8.png'),cv2.IMREAD_GRAYSCALE)
    clean=cv2.imread(str(CLEAN_MAP[str(page)]),cv2.IMREAD_GRAYSCALE)
    if observed is None or v78 is None or clean is None or not (observed.shape==v78.shape==clean.shape):
        raise RuntimeError(f'{page}: catastrophic-audit image read/shape failure')
    g=results[float(t)]['work']
    g=g[g.page.eq(str(page))]
    ys=g.y.to_numpy(int); xs=g.x.to_numpy(int); selected=g.decision.eq('PRINT').to_numpy()
    out=v78.copy(); out[ys[selected],xs[selected]]=observed[ys[selected],xs[selected]]
    return observed,v78,clean,out


def component_selected_points(page,t,cid):
    g=results[float(t)]['work']
    g=g[g.page.eq(str(page))].copy()
    if g.empty:
        return g
    component_ids=page_context[str(page)]['component_id'][g.y.to_numpy(int),g.x.to_numpy(int)]
    g['mask_component_id']=component_ids
    return g[g.decision.eq('PRINT') & g.mask_component_id.eq(int(cid))].copy()


def authorised_component_bbox(page,cid):
    stats=page_context[str(page)]['component_stats']
    cid=int(cid)
    if cid<=0 or cid>=len(stats):
        raise RuntimeError(f'{page}: invalid authoritative component ID {cid}')
    x,y,w,h,area=map(int,stats[cid])
    return x,y,x+w-1,y+h-1,area


def draw_hollow_box(image,bbox,colour=PURPLE,thickness=2,pad=5):
    out=cv2.cvtColor(image,cv2.COLOR_GRAY2BGR) if image.ndim==2 else image.copy()
    h,w=out.shape[:2]; x0,y0,x1,y1=map(int,bbox)
    cv2.rectangle(out,(max(0,x0-pad),max(0,y0-pad)),(min(w-1,x1+pad),min(h-1,y1+pad)),colour,thickness)
    return out


def truth_overlay(base,points,bbox):
    out=cv2.cvtColor(base,cv2.COLOR_GRAY2BGR)
    for truth,colour in TRUTH_BGR.items():
        q=points[points.truth.eq(truth)]
        if len(q):
            out[q.y.to_numpy(int),q.x.to_numpy(int)]=colour
    return draw_hollow_box(out,bbox)


def crop_bgr(image,bbox,pad):
    h,w=image.shape[:2]; x0,y0,x1,y1=map(int,bbox)
    xa=max(0,x0-pad); xb=min(w,x1+pad+1); ya=max(0,y0-pad); yb=min(h,y1+pad+1)
    return image[ya:yb,xa:xb],(xa,ya,xb,yb)


def comparison_panel(images,titles,subtitle=None):
    converted=[cv2.cvtColor(im,cv2.COLOR_GRAY2BGR) if im.ndim==2 else im for im in images]
    H=max(im.shape[0] for im in converted); W=max(im.shape[1] for im in converted); header=86
    canvas=np.full((H+header,W*len(converted),3),255,np.uint8)
    for i,(im,title) in enumerate(zip(converted,titles)):
        canvas[header:header+im.shape[0],i*W:i*W+im.shape[1]]=im
        cv2.putText(canvas,title,(i*W+8,26),cv2.FONT_HERSHEY_SIMPLEX,.54,(20,20,20),1,cv2.LINE_AA)
    if subtitle:
        cv2.putText(canvas,subtitle,(8,60),cv2.FONT_HERSHEY_SIMPLEX,.48,(20,20,20),1,cv2.LINE_AA)
    return canvas


def resize_max_width(image,max_width):
    if image.shape[1]<=max_width:
        return image
    scale=max_width/image.shape[1]
    return cv2.resize(image,(max_width,max(1,int(round(image.shape[0]*scale)))),interpolation=cv2.INTER_AREA)


# Identify every cumulative catastrophe and the first threshold at which each stable mask component becomes catastrophic.
cat_rows=[]
for t in SWEEP_THRESHOLDS:
    comps=results[t]['components']
    if len(comps):
        q=comps[comps.catastrophic].copy()
        if len(q): cat_rows.append(q)
cat_all=pd.concat(cat_rows,ignore_index=True) if cat_rows else pd.DataFrame(columns=[
    'policy_id','threshold','page','source_group','mask_component_id','selected','true_print',
    'false_annotation','truth_uncertain','utility_without_catastrophe','catastrophic',
    'xmin','xmax','ymin','ymax','mean_p_print','min_p_print'])
cat_all.to_csv(TABLES/'catastrophic_components_all_thresholds.csv',index=False)

first_rows=[]
for (page,cid),g in cat_all.groupby(['page','mask_component_id'],sort=False):
    # Earliest appearance while sweeping downward = highest threshold at which catastrophic first becomes true.
    first_t=max(float(x) for x in g.threshold)
    idx=SWEEP_THRESHOLDS.index(first_t)
    previous_t=SWEEP_THRESHOLDS[idx-1] if idx>0 else None
    row=g[g.threshold.eq(first_t)].iloc[0].to_dict()
    ax0,ay0,ax1,ay1,auth_area=authorised_component_bbox(page,cid)
    previous_cat=False
    if previous_t is not None:
        prev=results[previous_t]['components']
        hit=prev[prev.page.eq(str(page)) & prev.mask_component_id.eq(int(cid))]
        previous_cat=bool(hit.catastrophic.iloc[0]) if len(hit) else False
    if previous_cat:
        raise RuntimeError(f'{page}/{cid}: first-catastrophe transition is not actually a transition')
    row.update({
        'first_catastrophic_threshold':first_t,
        'previous_higher_threshold':previous_t,
        'previous_higher_was_catastrophic':previous_cat,
        'authoritative_xmin':ax0,'authoritative_ymin':ay0,
        'authoritative_xmax':ax1,'authoritative_ymax':ay1,
        'authoritative_component_area':auth_area,
        'false_true_ratio':float(row['false_annotation'])/max(int(row['true_print']),1),
        'catastrophic_reason':(
            'false_annotation>=50' if int(row['false_annotation'])>=50
            else 'false_true_ratio>1_and_false_annotation>=20'
        ),
    })
    first_rows.append(row)

first_cat=pd.DataFrame(first_rows)
if len(first_cat):
    first_cat=first_cat.sort_values(['first_catastrophic_threshold','false_annotation','selected'],ascending=[False,False,False]).reset_index(drop=True)
    first_cat['catastrophe_order']=np.arange(1,len(first_cat)+1)
    first_cat['visual_review']=''
    first_cat['review_note']=''
else:
    first_cat=pd.DataFrame(columns=[
        'catastrophe_order','page','source_group','mask_component_id','first_catastrophic_threshold',
        'previous_higher_threshold','selected','true_print','false_annotation','truth_uncertain',
        'false_true_ratio','catastrophic_reason','visual_review','review_note'])

first_cat.to_csv(TABLES/'catastrophic_component_first_appearance_inventory.csv',index=False)
review_path=REVIEW/'catastrophic_component_visual_review.csv'
first_cat.to_csv(review_path,index=False)
(REVIEW/'catastrophic_component_review_choices.json').write_text(json.dumps({
    'allowed_visual_review_values':CATASTROPHE_REVIEW_CHOICES,
    'instruction':'Enter one allowed value in visual_review after comparing the higher- and lower-threshold outputs; use clean/overlays only for diagnosis.'
},indent=2))

visual_inventory=[]; contact_files=[]
for _,r in first_cat.iterrows():
    page=str(r.page); cid=int(r.mask_component_id); lo=float(r.first_catastrophic_threshold)
    hi=None if pd.isna(r.previous_higher_threshold) else float(r.previous_higher_threshold)
    observed,v78,clean,lo_img=candidate_for_threshold(page,lo)
    if hi is None:
        hi_img=v78.copy(); hi_label='V7.8 baseline'
        hi_points=pd.DataFrame(columns=results[lo]['work'].columns)
    else:
        _,_,_,hi_img=candidate_for_threshold(page,hi); hi_label=f'Higher {hi:.3f}'
        hi_points=component_selected_points(page,hi,cid)
    lo_points=component_selected_points(page,lo,cid)
    hi_xy=set(zip(hi_points.y.astype(int),hi_points.x.astype(int))) if len(hi_points) else set()
    new_points=lo_points[[ (int(y),int(x)) not in hi_xy for y,x in zip(lo_points.y,lo_points.x) ]].copy()
    bbox=(int(r.authoritative_xmin),int(r.authoritative_ymin),int(r.authoritative_xmax),int(r.authoritative_ymax))
    obs_box=draw_hollow_box(observed,bbox); hi_box=draw_hollow_box(hi_img,bbox); lo_box=draw_hollow_box(lo_img,bbox); clean_box=draw_hollow_box(clean,bbox)
    cumulative_overlay=truth_overlay(observed,lo_points,bbox)
    transition_overlay=truth_overlay(observed,new_points,bbox)
    subtitle=(f'{page} component {cid} | first catastrophic at {lo:.3f} after '
              f'{hi if hi is not None else "baseline"} | selected={int(r.selected)} '
              f'TP={int(r.true_print)} FP={int(r.false_annotation)} U={int(r.truth_uncertain)} '
              f'FP/TP={float(r.false_true_ratio):.2f}')
    outdir=VISUAL/'catastrophic_components'/f'first_{threshold_slug(lo)}'
    outdir.mkdir(parents=True,exist_ok=True)
    stem=f'{int(r.catastrophe_order):03d}_{page}_component_{cid}'
    for context,pad in [('local',CATASTROPHE_LOCAL_PADDING),('wide',CATASTROPHE_WIDE_PADDING)]:
        crops=[]; crop_box_used=None
        for im in [obs_box,hi_box,lo_box,clean_box,cumulative_overlay,transition_overlay]:
            crop,crop_box_used=crop_bgr(im,bbox,pad); crops.append(crop)
        sheet=comparison_panel(crops,[
            'Observed',hi_label,f'Lower {lo:.3f}','Clean diagnostic',
            'Cumulative truth overlay','New-band truth overlay'],subtitle)
        path=outdir/f'{stem}_{context}.png'; cv2.imwrite(str(path),sheet)
        visual_inventory.append({
            'catastrophe_order':int(r.catastrophe_order),'page':page,'mask_component_id':cid,
            'first_catastrophic_threshold':lo,'previous_higher_threshold':hi,
            'context':context,'padding':pad,'crop_box':'|'.join(map(str,crop_box_used)),
            'file':str(path.relative_to(OUT))})
        if context=='wide': contact_files.append(path)
    # Whole-page locator: equal-scale higher and lower candidates with the same box.
    page_pair=comparison_panel([
        resize_max_width(hi_box,CATASTROPHE_PAGE_MAX_WIDTH),
        resize_max_width(lo_box,CATASTROPHE_PAGE_MAX_WIDTH)],
        [hi_label,f'First catastrophic {lo:.3f}'],subtitle)
    page_path=outdir/f'{stem}_whole_page_locator.png'; cv2.imwrite(str(page_path),page_pair)
    visual_inventory.append({
        'catastrophe_order':int(r.catastrophe_order),'page':page,'mask_component_id':cid,
        'first_catastrophic_threshold':lo,'previous_higher_threshold':hi,
        'context':'whole_page','padding':None,'crop_box':'whole-page',
        'file':str(page_path.relative_to(OUT))})

pd.DataFrame(visual_inventory).to_csv(TABLES/'catastrophic_component_visual_inventory.csv',index=False)

# Contact sheets are paginated to avoid one unusably tall image.
contact_dir=VISUAL/'catastrophic_components'
contact_dir.mkdir(parents=True,exist_ok=True)
for start in range(0,len(contact_files),CATASTROPHE_CONTACT_SHEET_ROWS):
    ims=[cv2.imread(str(p)) for p in contact_files[start:start+CATASTROPHE_CONTACT_SHEET_ROWS]]
    ims=[im for im in ims if im is not None]
    if not ims: continue
    width=max(im.shape[1] for im in ims); height=sum(im.shape[0] for im in ims)
    sheet=np.full((height,width,3),255,np.uint8); y=0
    for im in ims:
        sheet[y:y+im.shape[0],:im.shape[1]]=im; y+=im.shape[0]
    cv2.imwrite(str(contact_dir/f'catastrophic_component_contact_sheet_{start//CATASTROPHE_CONTACT_SHEET_ROWS+1:02d}.png'),sheet)

cat_summary={
    'catastrophic_rows_across_thresholds':int(len(cat_all)),
    'unique_components_ever_catastrophic':int(len(first_cat)),
    'first_truth_defined_catastrophe_threshold':float(first_cat.first_catastrophic_threshold.max()) if len(first_cat) else None,
    'review_choices':CATASTROPHE_REVIEW_CHOICES,
    'truth_overlay_colours':{'green':'PRINT','red':'ANNOTATION','amber':'UNCERTAIN'},
}
(OUT/'catastrophic_component_audit_summary.json').write_text(json.dumps(cat_summary,indent=2))
print(json.dumps(cat_summary,indent=2))
print('Catastrophic visual inventory:',TABLES/'catastrophic_component_visual_inventory.csv')

## 11. Final identity and executive summary

In [ ]:
# Optional lineage-only copy of the latest completed Notebook 36 result; never used in calculations.
notebook36_record=None
hits36=complete_runs(V82G_ROOT,['final_review_analysis.json','experiment_identity.json'])
if hits36:
    n36=hits36[-1]
    notebook36_record={
        'run':str(n36),
        'final_review_analysis_sha256':sha256_file(n36/'final_review_analysis.json'),
        'experiment_identity_sha256':sha256_file(n36/'experiment_identity.json'),
        'used_for_threshold_selection':False,
    }
    (AUDIT/'notebook36_lineage_only.json').write_text(json.dumps(notebook36_record,indent=2))

environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
(OUT/'software_environment.json').write_text(json.dumps(environment,indent=2))
identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.2h-downward-print-threshold-safe-coverage-frontier-catastrophic-audit-amendment',
    'development_only':True,'promotion_authority':False,
    'retrained':False,'refitted':False,'recalibrated':False,
    'source_v82_run':str(SOURCE),'source_record':source_record,
    'notebook36_lineage_only':notebook36_record,
    'sweep_thresholds':SWEEP_THRESHOLDS,
    'coordinated_margin_rule':'margin = max(0, 2*threshold - 1)',
    'annotation_threshold':ANNOTATION_THRESHOLD,
    'run_filter':ENABLE_RUN_FILTER,'minimum_print_run':MIN_PRINT_RUN,
    'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'bootstrap_seed':BOOTSTRAP_SEED,
    'render_thresholds':RENDER_THRESHOLDS,
    'threshold_frontier_sha256':sha256_file(TABLES/'threshold_frontier_summary.csv'),
    'incremental_summary_sha256':sha256_file(TABLES/'incremental_threshold_band_summary.csv'),
    'whole_page_integrity_sha256':sha256_file(AUDIT/'whole_page_integrity_and_hashes.csv'),
    'catastrophic_first_appearance_sha256':sha256_file(TABLES/'catastrophic_component_first_appearance_inventory.csv'),
    'catastrophic_visual_inventory_sha256':sha256_file(TABLES/'catastrophic_component_visual_inventory.csv'),
    'catastrophic_audit_summary_sha256':sha256_file(OUT/'catastrophic_component_audit_summary.json'),
    'software':environment,
}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

first_fail=[]
for _,r in sweep.sort_values('threshold',ascending=False).iterrows():
    failed=[k for k in ['D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability'] if not bool(r[k])]
    first_fail.append({'threshold':float(r.threshold),'failed_computable_gates':'|'.join(failed),'core_D3_D8':bool(r.core_development_envelope_D3_to_D8)})
pd.DataFrame(first_fail).to_csv(TABLES/'computable_gate_map.csv',index=False)

lines=[
    'NOTEBOOK 37 — DOWNWARD PRINT-THRESHOLD SAFE-COVERAGE FRONTIER','',
    f'Source V8.2 run: {SOURCE}',f'Notebook 37 run: {OUT}',
    'Retrained: False','Refitted: False','Recalibrated: False','Development only: True','',
    f'Thresholds: {SWEEP_THRESHOLDS}',
    f'Lowest threshold meeting computable D3-D8 envelope: {lowest_core}',
    f'Highest observed utility policy: {best_utility.get("policy_id")}',
    f'Highest observed utility: {best_utility.get("utility")}',
    f'Unique components ever catastrophic: {cat_summary["unique_components_ever_catastrophic"]}',
    f'First truth-defined catastrophic threshold: {cat_summary["first_truth_defined_catastrophe_threshold"]}',
    '',
    'Interpretation: retrospective development threshold sensitivity only.',
    'D1, D2, D9, D10, D11 and D12 are not fully adjudicated by the compact D3-D8 frontier flag.',
    'No threshold is frozen, independently validated or authorised for production use.',
    'The 0.995 policy replayed the registered V8.2 whole pages exactly.',
]
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines)); print('\nNOTEBOOK 37 COMPLETE:',OUT)

## Outputs to inspect first

1. `executive_summary.txt`
2. `tables/threshold_frontier_summary.csv`
3. `tables/incremental_threshold_band_summary.csv`
4. `tables/computable_gate_map.csv`
5. `visual_audit/threshold_frontier_true_false.png`
6. `visual_audit/threshold_frontier_utility.png`
7. `visual_audit/threshold_frontier_coverage_safety.png`
8. `review/threshold_increment_review_queue.csv`
9. `review/threshold_increment_crop_inventory.csv`
10. `whole_pages/<threshold>/<page>/threshold_candidate.png`
11. `whole_pages/<threshold>/<page>/newly_preserved_vs_0995_overlay.png`
12. `audit/whole_page_integrity_and_hashes.csv`
13. `tables/catastrophic_component_first_appearance_inventory.csv`
14. `tables/catastrophic_component_visual_inventory.csv`
15. `review/catastrophic_component_visual_review.csv`
16. `visual_audit/catastrophic_components/catastrophic_component_contact_sheet_*.png`
17. `visual_audit/catastrophic_components/first_<threshold>/*_whole_page_locator.png`

The key decision is not “which row has the largest utility?” It is:

> How far can the threshold be lowered before the newly added band introduces objectionable annotation, catastrophic concentration, source-group instability or a material safety-bound failure?